In [17]:
import torch
import torch.nn.functional as F
from einops import rearrange, reduce
from typing import Dict, Tuple
from torch import Tensor
from typing_extensions import Annotated as Ann


def calc_seq_vcr_loss(
    hs: Ann[Tensor, "b t h"], 
    lambda1: float = 1.0,
    lambda2: float = 0.1,
    eta: float = 1e-3
) -> Tuple[Ann[Tensor, ""], Dict[str, float]]:
    """
    Calculate Sequential Variance-Covariance Regularization Loss.
    
    Args:
        hs: Hidden states tensor of shape [batch_size, seq_len, hidden_dim]
        lambda1: Coefficient for variance term (default: 1.0)
        lambda2: Coefficient for covariance term (default: 1.0)
        eta: Small constant for numerical stability (default: 1e-3)
        
    Returns:
        Tuple containing:
            - Scalar loss value
            - Dictionary with individual loss components
    """
    # Get dimensions
    batch_size, seq_len, hidden_dim = hs.shape
    
    if batch_size <= 1:
        raise ValueError("Batch size must be greater than 1 to calculate covariance")
    
    # Initialize loss components
    var_loss_total = 0.0
    cov_loss_total = 0.0
    
    # Loop over each position in the sequence
    for i in range(seq_len):
        # Get representations for position i across all batches
        pos_repr = hs[:, i, :]  # Shape: [batch_size, hidden_dim]
        
        # Calculate mean across batch dimension
        mean = torch.mean(pos_repr, dim=0, keepdim=True)  # Shape: [1, hidden_dim]
        
        # Center the data
        centered = pos_repr - mean  # Shape: [batch_size, hidden_dim]
        
        # Calculate covariance matrix (match paper's equation 4)
        cov = torch.matmul(centered.T, centered) / (batch_size - 1)  # Shape: [hidden_dim, hidden_dim]
        
        # Variance term: sum of max(0, 1-sqrt(C_{i,k,k}+eta))
        var_diag = torch.diag(cov)  # Get diagonal elements (variances)
        var_term = torch.sum(F.relu(1.0 - torch.sqrt(var_diag + eta)))
        
        # Covariance term: sum of (C_{i,k,k'})^2 for k != k'
        # Create a mask to zero out the diagonal elements
        mask = torch.ones_like(cov) - torch.eye(hidden_dim, device=cov.device)
        cov_off_diag = cov * mask
        cov_term = torch.sum(cov_off_diag ** 2)
        
        # Add to total loss
        var_loss_total += var_term
        cov_loss_total += cov_term
    
    # Normalize by sequence length and feature dimension (match paper's equation 3)
    var_loss_normalized = var_loss_total / (seq_len * hidden_dim)
    cov_loss_normalized = cov_loss_total / (seq_len * hidden_dim)
    
    # Calculate total loss
    loss = lambda1 * var_loss_normalized + lambda2 * cov_loss_normalized
    
    return loss, {
        "loss_vcr_var": var_loss_normalized.item(),
        "loss_vcr_cov": cov_loss_normalized.item()
    }

# Alternative implementation using einops for more concise tensor manipulation
def calc_seq_vcr_loss_einops(
    hs: Ann[Tensor, "b t h"], 
    λ1: float = 1.0,
    λ2: float = 0.1,
    η: float = 1e-3
) -> Tuple[Ann[Tensor, ""], Dict[str, float]]:
    """
    Calculate Sequential Variance-Covariance Regularization Loss using einops.
    
    Args:
        hs: Hidden states tensor of shape [batch_size, seq_len, hidden_dim]
        lambda1: Coefficient for variance term (default: 1.0)
        lambda2: Coefficient for covariance term (default: 1.0)
        eta: Small constant for numerical stability (default: 1e-3)
        
    Returns:
        Tuple containing:
            - Scalar loss value
            - Dictionary with individual loss components
    """
    B, T, P = hs.shape
    
    if B <= 1:
        raise ValueError("Batch size must be greater than 1 to calculate covariance")
    
    # Transpose to [seq_len, batch_size, hidden_dim] for per-timestep operations
    x = rearrange(hs, "b t h -> t b h")
    
    # Calculate mean across batch dimension for each timestep
    x_mean = x.mean(dim=1, keepdim=True)  # Shape: [seq_len, 1, hidden_dim]
    
    # Center the data
    x_centered = x - x_mean  # Shape: [seq_len, batch_size, hidden_dim]
    
    # Calculate covariance matrix for each timestep
    # Batch matrix multiplication: for each t, multiply x_centered[t].T * x_centered[t]
    # Shape: [seq_len, hidden_dim, hidden_dim]
    C = torch.bmm(
        rearrange(x_centered, "t b h -> t h b"),  # [seq_len, hidden_dim, batch_size]
        rearrange(x_centered, "t b h -> t b h")   # [seq_len, batch_size, hidden_dim]
    ) / (B - 1)
    
    # Create diagonal mask
    diag_mask = torch.eye(P, device=hs.device).bool()
    off_diag_mask = ~diag_mask
    
    # Extract diagonal elements (variances) for each timestep
    # Shape: [seq_len, hidden_dim]
    diag = torch.diagonal(C, dim1=1, dim2=2)
    
    # Calculate variance term
    var_term = F.relu(1.0 - torch.sqrt(diag + η))  # Shape: [seq_len, hidden_dim]
    var_loss = reduce(var_term, "t h -> ", "sum") / (T * P)
    
    # Calculate covariance term
    # For each timestep, zero out diagonal elements and square the rest
    cov_term = torch.zeros_like(C)
    for t in range(T):
        cov_term[t] = C[t] * off_diag_mask  # Zero out diagonal
    
    cov_term = cov_term ** 2  # Square the covariance values
    cov_loss = reduce(cov_term, "t h1 h2 -> ", "sum") / (T * P)
    
    # Calculate total loss
    loss = λ1 * var_loss + λ2 * cov_loss
    
    return loss, {
        "loss_vcr_var": var_loss.item(),
        "loss_vcr_cov": cov_loss.item()
    }

from jaxtyping import Float
from torch import Tensor

def calc_seq_vcr_loss2(hs: Float[Tensor, "b t h"], η = 1e-3, λ1=1, λ2=0.1) -> Float[Tensor, ""]:
    """
    Calculate Sequential Variance-Covariance Regularization Loss using einops.
    
    Args:
        hs: Hidden states tensor of shape [batch_size, seq_len, hidden_dim]
        lambda1: Coefficient for variance term (default: 1.0)
        lambda2: Coefficient for covariance term (default: 1.0)
        eta: Small constant for numerical stability (default: 1e-3)
        
    Returns:
        Tuple containing:
            - Scalar loss value
            - Dictionary with individual loss components
    """
    B, T, P = hs.shape

    if B <= 1:
        raise ValueError("Batch size must be greater than 1 to calculate covariance")

    # Compute covariance per timestep across batch
    x = rearrange(hs, "b t h -> t b h")  # Shape: (T, B, P)

    # Calculate mean and center per timestep x.dtype
    x_mean = x.mean(dim=1, keepdim=True)#.detach()  # Shape: (T, 1, P)
    x_centered = x - x_mean

    # Compute covariance matrices for each timestep
    C = torch.bmm(x_centered.transpose(1, 2), x_centered) / (B - 1)  # Shape: (T, P, P)

    # Setup mask for diagonal elements
    diag = torch.eye(P, dtype=torch.bool, device=x.device)#.detach()
    C_diag = torch.diagonal(C, dim1=1, dim2=2)
    
    # The Variance Term encourages unit variance in each dimension (most important)
    var_loss = torch.relu(1 - torch.sqrt(C_diag + η))
    var_loss = reduce(var_loss, "t h1 -> ", "sum") / (T * P)

    # the Covariance Term penalizes covariance between different dimensions, promoting decorrelation and diversity in representations
    non_diag = (~diag).detach()
    cov_loss = (C * non_diag).pow(2)
    cov_loss = reduce(cov_loss, "t h1 h2 -> ", "sum") / (T * P)

    # Combine and reduce
    loss = λ1 * var_loss + λ2 * cov_loss
    return loss, {"loss_vcr_var": var_loss.item(), "loss_vcr_cov": cov_loss.item()}

In [22]:
# Create random tensor for testing
B, T, P = 16, 10, 32
hs = torch.randn(B, T, P) 

# Calculate loss using both implementations
loss1, metrics1 = calc_seq_vcr_loss(hs)
loss2, metrics2 = calc_seq_vcr_loss_einops(hs)
loss3, metrics3 = calc_seq_vcr_loss2(hs)

print("Standard implementation:")
print(f"  Total loss: {loss1.item():.6f}")
print(f"  Variance loss: {metrics1['loss_vcr_var']:.6f}")
print(f"  Covariance loss: {metrics1['loss_vcr_cov']:.6f}")

print("\nEinops implementation:")
print(f"  Total loss: {loss2.item():.6f}")
print(f"  Variance loss: {metrics2['loss_vcr_var']:.6f}")
print(f"  Covariance loss: {metrics2['loss_vcr_cov']:.6f}")

print("\nAlternative implementation:")
print(f"  Total loss: {loss3.item():.6f}")
print(f"  Variance loss: {metrics3['loss_vcr_var']:.6f}")
print(f"  Covariance loss: {metrics3['loss_vcr_cov']:.6f}")

# Check if both implementations give similar results
assert torch.isclose(loss1, loss2, rtol=1e-4), "Implementations give different results"
print("\nBoth implementations give similar results! ✓")

Standard implementation:
  Total loss: 0.280484
  Variance loss: 0.086057
  Covariance loss: 1.944266

Einops implementation:
  Total loss: 0.280484
  Variance loss: 0.086057
  Covariance loss: 1.944267

Alternative implementation:
  Total loss: 0.280484
  Variance loss: 0.086057
  Covariance loss: 1.944267

Both implementations give similar results! ✓
